# Week 4: Semantic Search vs BM25

Goal:
- compare semantic search and BM25
- check latency on 10k listings
- do 50 query-result relevance check


In [9]:
import os
import sys
import csv
import json
import math
import re
import time
import statistics

sys.path.append(os.path.abspath('..'))
from scripts.semantic_search import SemanticSearcher


In [10]:
# load data
CSV_PATH = '../data/processed/cleaned_listing_sample.csv'
QUERY_PATH = '../data/processed/user_queries.json'
TOP_K = 10
EMBEDDING_DIM = 384  # can switch to 768

rows = []
with open(CSV_PATH, 'r', encoding='utf-8', newline='') as f:
    reader = csv.DictReader(f)
    for i, r in enumerate(reader):
        txt = (r.get('remarks_after_cleaning') or '').strip()
        if not txt:
            txt = (r.get('remarks') or '').strip()
        if not txt:
            continue
        try:
            price = float(str(r.get('price', '0')).replace(',', '').strip() or 0)
        except ValueError:
            price = 0.0
        rows.append({
            'idx': i,
            'listing_id': str(r.get('L_ListingID', '')),
            'city': str(r.get('L_City', '')).strip(),
            'price': price,
            'remarks': txt,
        })

with open(QUERY_PATH, 'r', encoding='utf-8') as f:
    raw_queries = json.load(f)

queries = []
seen = set()
for x in raw_queries:
    q = str(x.get('query', '')).strip()
    if q and q not in seen:
        seen.add(q)
        queries.append(q)

print('listings:', len(rows))
print('queries:', len(queries))


listings: 1000
queries: 230


In [11]:
# simple BM25
TOKEN_RE = re.compile(r'[a-z0-9]+')

class BM25Searcher:
    def __init__(self, k1=1.5, b=0.75):
        self.k1 = k1
        self.b = b

    def _tok(self, s):
        return TOKEN_RE.findall((s or '').lower())

    def fit(self, docs):
        self.docs = list(docs)
        self.doc_tokens = [self._tok(d) for d in self.docs]
        self.doc_len = [len(t) for t in self.doc_tokens]
        self.avgdl = sum(self.doc_len) / max(1, len(self.doc_len))

        self.postings = {}
        self.df = {}
        for i, toks in enumerate(self.doc_tokens):
            tf = {}
            for t in toks:
                tf[t] = tf.get(t, 0) + 1
            for term, cnt in tf.items():
                self.postings.setdefault(term, []).append((i, cnt))
            for term in tf:
                self.df[term] = self.df.get(term, 0) + 1

        self.n_docs = len(self.docs)

    def _idf(self, term):
        df = self.df.get(term, 0)
        if df == 0:
            return 0.0
        return math.log(1.0 + (self.n_docs - df + 0.5) / (df + 0.5))

    def search(self, query, top_k=10):
        q_terms = self._tok(query)
        scores = {}
        for term in q_terms:
            idf = self._idf(term)
            if idf <= 0:
                continue
            for idx, tf in self.postings.get(term, []):
                dl = self.doc_len[idx]
                denom = tf + self.k1 * (1 - self.b + self.b * dl / max(1e-9, self.avgdl))
                s = idf * (tf * (self.k1 + 1)) / max(1e-9, denom)
                scores[idx] = scores.get(idx, 0.0) + s
        return sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_k]


In [12]:
# build both searchers
remarks = [r['remarks'] for r in rows]

semantic = SemanticSearcher(embedding_dim=EMBEDDING_DIM)
semantic.build_index(remarks)

bm25 = BM25Searcher()
bm25.fit(remarks)

print('index built')


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


index built


In [13]:
# relevance rules (lightweight heuristic)
PROPERTY_TYPES = ['single family home', 'townhouse', 'penthouse', 'apartment', 'duplex', 'condo', 'loft']
AMENITIES = ['pool', 'garage', 'fireplace', 'balcony', 'gym', 'solar', 'hardwood floors', 'smart home features', 'large backyard']
KNOWN_CITIES = {r['city'] for r in rows if r['city']}

def parse_money(raw):
    t = raw.lower().replace('$', '').replace(',', '').strip()
    m = 1.0
    if t.endswith('million'):
        m, t = 1_000_000.0, t[:-7].strip()
    elif t.endswith('m'):
        m, t = 1_000_000.0, t[:-1].strip()
    elif t.endswith('k'):
        m, t = 1_000.0, t[:-1].strip()
    return float(t) * m

def parse_filters(q):
    s = q.lower()
    city = None
    for c in sorted(KNOWN_CITIES, key=len, reverse=True):
        if re.search(rf'\b(in|near|around|at)\s+{re.escape(c.lower())}\b', s):
            city = c
            break

    ptype = next((p for p in PROPERTY_TYPES if p in s), None)

    price_max = None
    m1 = re.search(r'(?:under|below|up to|at most|no more than)\s+([\$\d.,]+\s*(?:k|m|million)?)', s)
    if m1:
        price_max = parse_money(m1.group(1))

    price_min = None
    m2 = re.search(r'(?:over|above|at least)\s+([\$\d.,]+\s*(?:k|m|million)?)', s)
    if m2:
        price_min = parse_money(m2.group(1))

    ams = [a for a in AMENITIES if a in s]

    return city, ptype, price_min, price_max, ams

def is_relevant(query, row):
    city, ptype, price_min, price_max, ams = parse_filters(query)
    txt = row['remarks'].lower()

    if city and row['city'].lower() != city.lower():
        return 0
    if price_min is not None and row['price'] < price_min:
        return 0
    if price_max is not None and row['price'] > price_max:
        return 0
    if ptype and ptype not in txt:
        return 0
    for a in ams:
        if a not in txt:
            return 0

    if not any([city, ptype, price_min is not None, price_max is not None, ams]):
        return 0

    return 1


In [14]:
# quality comparison
def precision_at_k(labels, k):
    x = labels[:k]
    return (sum(x) / len(x)) if x else 0.0

def mrr_at_k(labels, k):
    for i, v in enumerate(labels[:k], start=1):
        if v:
            return 1.0 / i
    return 0.0

test_queries = queries[:100]
sem_p, sem_mrr = [], []
bm_p, bm_mrr = [], []

for q in test_queries:
    sem_hits = semantic.search_with_indices(q, top_k=TOP_K)
    bm_hits = bm25.search(q, top_k=TOP_K)

    sem_labels = [is_relevant(q, rows[h.index]) for h in sem_hits]
    bm_labels = [is_relevant(q, rows[i]) for i, _ in bm_hits]

    sem_p.append(precision_at_k(sem_labels, TOP_K))
    sem_mrr.append(mrr_at_k(sem_labels, TOP_K))
    bm_p.append(precision_at_k(bm_labels, TOP_K))
    bm_mrr.append(mrr_at_k(bm_labels, TOP_K))

print('=== Quality (first 100 queries) ===')
print(f'Semantic Precision@{TOP_K}: {sum(sem_p)/len(sem_p):.4f}')
print(f'BM25    Precision@{TOP_K}: {sum(bm_p)/len(bm_p):.4f}')
print(f'Semantic MRR@{TOP_K}:      {sum(sem_mrr)/len(sem_mrr):.4f}')
print(f'BM25    MRR@{TOP_K}:      {sum(bm_mrr)/len(bm_mrr):.4f}')


=== Quality (first 100 queries) ===
Semantic Precision@10: 0.4970
BM25    Precision@10: 0.3260
Semantic MRR@10:      0.6714
BM25    MRR@10:      0.4472


In [15]:
# latency on 10k
rows_10k = []
i = 0
while len(rows_10k) < 10000:
    r = rows[i % len(rows)]
    rows_10k.append({
        'idx': len(rows_10k),
        'listing_id': f"{r['listing_id']}_{len(rows_10k)}",
        'city': r['city'],
        'price': r['price'],
        'remarks': r['remarks'],
    })
    i += 1

remarks_10k = [r['remarks'] for r in rows_10k]
semantic_10k = SemanticSearcher(embedding_dim=EMBEDDING_DIM)
semantic_10k.build_index(remarks_10k)

bm25_10k = BM25Searcher()
bm25_10k.fit(remarks_10k)

bench_queries = queries[:50]

sem_ms = []
for q in bench_queries:
    t0 = time.perf_counter()
    semantic_10k.search_with_indices(q, top_k=10)
    sem_ms.append((time.perf_counter() - t0) * 1000)

bm_ms = []
for q in bench_queries:
    t0 = time.perf_counter()
    bm25_10k.search(q, top_k=10)
    bm_ms.append((time.perf_counter() - t0) * 1000)

def stat(ms):
    p95 = sorted(ms)[int(0.95 * (len(ms)-1))]
    return {'avg': sum(ms)/len(ms), 'p50': statistics.median(ms), 'p95': p95, 'max': max(ms)}

sem_stat = stat(sem_ms)
bm_stat = stat(bm_ms)

print('=== Latency on 10k (ms) ===')
print('Semantic:', sem_stat)
print('BM25:    ', bm_stat)
print('Semantic p95 < 100ms:', sem_stat['p95'] < 100)
print('BM25 p95 < 100ms:', bm_stat['p95'] < 100)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


=== Latency on 10k (ms) ===
Semantic: {'avg': 17.326969113200903, 'p50': 8.181812707334757, 'p95': 25.45883320271969, 'max': 384.6004162915051}
BM25:     {'avg': 9.77105338126421, 'p50': 9.028416592627764, 'p95': 15.126291196793318, 'max': 51.46049987524748}
Semantic p95 < 100ms: True
BM25 p95 < 100ms: True


In [16]:
# 50 query-result pairs
pairs = []
for q in queries[:25]:
    s_hits = semantic.search_with_indices(q, top_k=1)
    b_hits = bm25.search(q, top_k=1)

    if s_hits:
        row = rows[s_hits[0].index]
        pairs.append({
            'query': q,
            'method': 'semantic',
            'listing_id': row['listing_id'],
            'city': row['city'],
            'score': s_hits[0].score,
            'relevance': is_relevant(q, row),
        })

    if b_hits:
        idx, score = b_hits[0]
        row = rows[idx]
        pairs.append({
            'query': q,
            'method': 'bm25',
            'listing_id': row['listing_id'],
            'city': row['city'],
            'score': score,
            'relevance': is_relevant(q, row),
        })

pairs = pairs[:50]

sem_hit = sum(x['relevance'] for x in pairs if x['method'] == 'semantic')
sem_total = sum(1 for x in pairs if x['method'] == 'semantic')
bm_hit = sum(x['relevance'] for x in pairs if x['method'] == 'bm25')
bm_total = sum(1 for x in pairs if x['method'] == 'bm25')

print('total pairs:', len(pairs))
print('semantic hit rate:', sem_hit / max(1, sem_total))
print('bm25 hit rate:', bm_hit / max(1, bm_total))

OUT = '../data/processed/relevance_eval_50_pairs_notebook.json'
with open(OUT, 'w', encoding='utf-8') as f:
    json.dump(pairs, f, ensure_ascii=False, indent=2)
print('saved:', OUT)


total pairs: 50
semantic hit rate: 0.56
bm25 hit rate: 0.2
saved: ../data/processed/relevance_eval_50_pairs_notebook.json
